# NFL intro — sportsdataverse-py

The NFL submodule mirrors [nflreadpy](https://github.com/nflverse/nflreadpy) so existing nflverse code can swap engines with minimal changes. Backed by [nflverse](https://nflverse.nflverse.com) parquet releases.

R companion: [nflfastR](https://www.nflfastr.com) / [nflverse](https://nflverse.nflverse.com).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse.nfl as nfl

## Caching layer

sdv-py NFL caches loader results to keep repeat calls fast. For reproducibility in a notebook, turn caching off; clean up afterwards.

In [ ]:
nfl.update_config(cache_mode='off')
nfl.get_config()

## nflreadpy parity surface — `load_pbp`, `load_schedules`

In [ ]:
schedules = nfl.load_schedules([2024])
schedules.shape

In [ ]:
(schedules
    .select(['season', 'week', 'gameday', 'home_team', 'away_team', 'home_score', 'away_score'])
    .head())

In [ ]:
pbp = nfl.load_pbp([2024])
pbp.shape

In [ ]:
(pbp
    .select(['game_id', 'play_id', 'qtr', 'down', 'ydstogo', 'desc'])
    .head())

## NextGen Stats

In [ ]:
ngs = nfl.load_nextgen_stats([2024], stat_type='passing')
ngs.shape

In [ ]:
(ngs
    .select(['season', 'player_display_name', 'team_abbr', 'attempts', 'pass_yards', 'completion_percentage_above_expectation'])
    .head())

## Current-season helpers

In [ ]:
nfl.get_current_season(), nfl.get_current_week()

## Schedule via ESPN (live, scoreboard-style)

In [ ]:
espn_sched = nfl.espn_nfl_schedule(dates=20240908)
espn_sched.select(['id', 'home_team_full_name', 'away_team_full_name', 'home_score', 'away_score']).head()

## Pipeline example: total points per team in 2024

Sum home and away scores for every team across the season.

In [ ]:
home = schedules.select(['home_team', 'home_score']).rename({'home_team': 'team', 'home_score': 'pts'})
away = schedules.select(['away_team', 'away_score']).rename({'away_team': 'team', 'away_score': 'pts'})
totals = (
    pl.concat([home, away])
    .drop_nulls('pts')
    .group_by('team')
    .agg(pl.col('pts').sum().alias('points_for'),
         pl.len().alias('games'))
    .sort('points_for', descending=True)
)
totals.head(10)

## Clean up the cache

In [ ]:
nfl.clear_cache()

## Cross-references

- nflverse: <https://nflverse.nflverse.com>
- nflreadpy (Python): <https://github.com/nflverse/nflreadpy>
- nflfastR (R): <https://www.nflfastr.com>
- Plotting: matplotlib, plotnine

## Where to go next

- API docs: `docs/docs/nfl/index.md`
- Next notebook: `04_nba_intro.ipynb`